# 🤟 Indian Sign Language Translation — Video to Text
**Dataset:** ISL-CSLTR Corpus (ISL_CSLRT_Corpus)
**Stack:** Python 3.10 · MediaPipe · PyTorch · Gradio

---
### Real dataset structure this notebook expects
```
ISL_CSLRT_Corpus/
├── Frames_Sentence_Level/     ← pre-extracted frames per sentence (USE THIS)
├── Frames_Word_Level/         ← pre-extracted frames per word
├── Videos_Sentence_Level/    ← raw .mp4 videos (alternative source)
└── corpus_csv_files/
    ├── ISL_CSLRT_Corpus_*.csv ← annotations (signer, label, paths)
    └── ISL_CSLRT.txt
```

### Run order
| Section | What it does |
|---|---|
| 1 | Install dependencies |
| 2 | Config + explore dataset structure |
| 3 | Read CSV annotations → map frames to labels |
| 4 | Run MediaPipe on frames → save keypoints (.npy) |
| 5 | PyTorch Dataset & DataLoader |
| 6 | Model (BiLSTM with attention) |
| 7 | Train |
| 8 | Evaluate |
| 9 | Real-time inference + Gradio UI |


## Section 1 — Install dependencies
Run once. Restart kernel if asked.

In [ ]:
import subprocess, sys

packages = [
    "torch torchvision --index-url https://download.pytorch.org/whl/cpu",
    "opencv-python",
    "mediapipe==0.10.11",
    "scikit-learn",
    "tqdm",
    "jiwer",
    "gradio",
    "matplotlib",
    "pandas",
    "pyyaml",
    "seaborn",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *pkg.split()], check=True)

print("\n✅ All packages installed!")


## Section 2 — Config & explore dataset structure
**Edit `DATASET_ROOT` below to point to your `ISL_CSLRT_Corpus` folder.**


In [ ]:
import os, json, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ─── EDIT THIS ────────────────────────────────────────────────────────────────
DATASET_ROOT = r"C:/Users/YourName/Downloads/ISL_CSLRT_Corpus"   # ← your path
# ──────────────────────────────────────────────────────────────────────────────

FRAMES_SENTENCE = os.path.join(DATASET_ROOT, "Frames_Sentence_Level")
FRAMES_WORD     = os.path.join(DATASET_ROOT, "Frames_Word_Level")
VIDEOS          = os.path.join(DATASET_ROOT, "Videos_Sentence_Level")
CSV_DIR         = os.path.join(DATASET_ROOT, "corpus_csv_files")
PROCESSED_DIR   = "data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs("checkpoints",  exist_ok=True)

CFG = {
    "max_seq_len":    150,
    "num_keypoints":  1662,   # 33×4 pose + 468×3 face + 21×3 lh + 21×3 rh
    "hidden_size":    256,
    "num_layers":     3,
    "dropout":        0.4,
    "epochs":         80,
    "batch_size":     16,
    "learning_rate":  3e-4,
    "patience":       15,
    "device":         "cuda",  # change to "cpu" if no GPU
    "checkpoint":     "checkpoints/best_model.pt",
    "label_map":      "data/processed/label_map.json",
    "samples":        "data/processed/samples.json",
}

# ── Explore top-level folders ──
print("=== Dataset root contents ===")
for item in sorted(os.listdir(DATASET_ROOT)):
    full = os.path.join(DATASET_ROOT, item)
    if os.path.isdir(full):
        sub_count = len(os.listdir(full))
        print(f"  📁 {item}/  ({sub_count} items)")
    else:
        print(f"  📄 {item}")

# ── Explore CSV files ──
print("\n=== CSV files found ===")
csv_files = list(Path(CSV_DIR).glob("*.csv")) + list(Path(CSV_DIR).glob("*.CSV"))
for f in csv_files:
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")

print(f"\n=== Frames_Sentence_Level top folders ===")
if os.path.exists(FRAMES_SENTENCE):
    folders = sorted(os.listdir(FRAMES_SENTENCE))[:20]
    for f in folders:
        sub = os.path.join(FRAMES_SENTENCE, f)
        if os.path.isdir(sub):
            n = len(os.listdir(sub))
            print(f"  {f}/  ({n} items)")


### Preview all CSV files — understand what columns exist

In [ ]:
import pandas as pd

all_dfs = {}
for csv_path in sorted(Path(CSV_DIR).glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
        all_dfs[csv_path.name] = df
        print(f"\n{'─'*60}")
        print(f"FILE : {csv_path.name}")
        print(f"SHAPE: {df.shape}  (rows × cols)")
        print(f"COLS : {list(df.columns)}")
        print(df.head(3).to_string())
    except Exception as e:
        print(f"Could not read {csv_path.name}: {e}")


## Section 3 — Parse CSV annotations → build sample list
This section reads the CSV files and builds a clean list of  
`{ frame_folder, label }` pairs that Section 4 will process.

> **Run Section 2 first** — look at the CSV column names printed above,  
> then confirm or update the column name variables below.


In [ ]:
import pandas as pd
from pathlib import Path

# ─── After looking at Section 2 output, set the correct column names ──────────
COL_LABEL      = "Gloss"           # column with the sign label / sentence text
COL_SIGNER     = "Signer_ID"       # signer identifier column
COL_FRAME_FOLD = "Folder_Name"     # column pointing to the frame subfolder (if any)
# (Common alternatives: "label","sentence","sign","class","Sign_ID","Sentence_ID")
# ──────────────────────────────────────────────────────────────────────────────

def load_best_csv():
    """Load the most informative CSV (prefer the largest one)."""
    csvs = sorted(Path(CSV_DIR).glob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSVs found in {CSV_DIR}")
    print(f"Using: {csvs[0].name}")
    return pd.read_csv(csvs[0])

df = load_best_csv()
print(f"Shape: {df.shape}")
print(df.head(5))
print(f"\nAll columns: {list(df.columns)}")


In [ ]:
# ── Auto-detect or manually confirm column names ──────────────────────────────
# Update these to match what was printed above
LABEL_COL  = COL_LABEL     # ← change if needed
SIGNER_COL = COL_SIGNER    # ← change if needed

# Check columns exist
for col in [LABEL_COL]:
    if col not in df.columns:
        print(f"⚠ Column '{col}' not found! Available: {list(df.columns)}")
    else:
        unique_labels = df[LABEL_COL].dropna().unique()
        print(f"✅ Label column  : '{LABEL_COL}' — {len(unique_labels)} unique labels")
        print(f"   Sample labels : {list(unique_labels[:10])}")

if SIGNER_COL in df.columns:
    print(f"✅ Signer column : '{SIGNER_COL}' — {df[SIGNER_COL].nunique()} signers")
    print(f"   Signer IDs    : {sorted(df[SIGNER_COL].unique())}")

# Class distribution
plt.figure(figsize=(14, 3))
df[LABEL_COL].value_counts().plot(kind="bar", color="#6366F1", alpha=0.85)
plt.title("Sample count per label")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# ── Map each row to its frame folder on disk ──────────────────────────────────
#
# Strategy: The Frames_Sentence_Level folder likely has sub-folders named by
# signer_id, sentence_id, or a combined key from the CSV.
# We try several matching strategies automatically.

def find_frame_folders(df, frames_root, label_col):
    """
    Tries to match each CSV row to a frame folder.
    Returns list of dicts: {path, label, num_frames}
    """
    samples  = []
    missing  = 0
    all_dirs = {d.name.lower(): d for d in Path(frames_root).iterdir() if d.is_dir()}

    print(f"Total frame sub-folders found: {len(all_dirs)}")
    print(f"Sample folder names          : {list(all_dirs.keys())[:8]}")

    for _, row in df.iterrows():
        label = str(row[label_col]).strip()

        # Strategy 1: folder name = label directly
        key1 = label.lower().replace(" ", "_")
        if key1 in all_dirs:
            folder = all_dirs[key1]
            imgs   = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
            if imgs:
                samples.append({"path": str(folder), "label": label, "num_frames": len(imgs)})
                continue

        # Strategy 2: use COL_FRAME_FOLD column if it exists
        if COL_FRAME_FOLD in row.index:
            key2 = str(row[COL_FRAME_FOLD]).strip().lower()
            if key2 in all_dirs:
                folder = all_dirs[key2]
                imgs   = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
                if imgs:
                    samples.append({"path": str(folder), "label": label, "num_frames": len(imgs)})
                    continue

        # Strategy 3: signer + row index as combined key
        if SIGNER_COL in row.index:
            key3 = f"{row[SIGNER_COL]}_{label}".lower().replace(" ", "_")
            if key3 in all_dirs:
                folder = all_dirs[key3]
                imgs   = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
                if imgs:
                    samples.append({"path": str(folder), "label": label, "num_frames": len(imgs)})
                    continue

        missing += 1

    print(f"\nMatched  : {len(samples)}")
    print(f"Not found: {missing}")
    return samples

raw_samples = find_frame_folders(df, FRAMES_SENTENCE, LABEL_COL)

if not raw_samples:
    print("\n⚠ Auto-matching failed. Running folder-scan fallback...")
    print("  Listing all frame folders and their parent structure:")
    for p in sorted(Path(FRAMES_SENTENCE).rglob("*.jpg"))[:5]:
        print(" ", p)
    print("\n  → Look at the paths above and update the matching logic in find_frame_folders()")
else:
    print(f"\n✅ {len(raw_samples)} sample sequences ready for keypoint extraction.")
    avg_frames = sum(s["num_frames"] for s in raw_samples) / len(raw_samples)
    print(f"   Avg frames per sequence: {avg_frames:.1f}")


In [ ]:
# ── If auto-match failed: MANUAL FALLBACK ─────────────────────────────────────
# Walk Frames_Sentence_Level and use folder names directly as labels.
# Use this cell ONLY if the previous cell showed 0 matches.

USE_FALLBACK = False   # ← set True if auto-match returned 0

if USE_FALLBACK:
    raw_samples = []
    for label_folder in sorted(Path(FRAMES_SENTENCE).iterdir()):
        if not label_folder.is_dir():
            continue
        label = label_folder.name

        # Each sub-folder = one signer's recording for this label
        sub_folders = [s for s in label_folder.iterdir() if s.is_dir()]
        if sub_folders:
            for sf in sub_folders:
                imgs = list(sf.glob("*.jpg")) + list(sf.glob("*.png"))
                if imgs:
                    raw_samples.append({"path": str(sf), "label": label,
                                        "num_frames": len(imgs)})
        else:
            # frames are directly inside label_folder
            imgs = list(label_folder.glob("*.jpg")) + list(label_folder.glob("*.png"))
            if imgs:
                raw_samples.append({"path": str(label_folder), "label": label,
                                    "num_frames": len(imgs)})

    print(f"Fallback found {len(raw_samples)} sequences across "
          f"{len(set(s['label'] for s in raw_samples))} classes.")


## Section 4 — MediaPipe keypoint extraction from frames
Since the dataset already has **pre-extracted frames**, we run MediaPipe directly  
on the images (much faster than processing videos frame-by-frame).  
Output: one `.npy` file per sequence saved to `data/processed/`.


In [ ]:
import cv2, numpy as np, mediapipe as mp
from tqdm.notebook import tqdm
import json, os

mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    """Flatten all MediaPipe landmarks → 1662-d vector."""
    pose = np.array([[r.x, r.y, r.z, r.visibility]
                     for r in results.pose_landmarks.landmark]).flatten()            if results.pose_landmarks else np.zeros(33 * 4)
    face = np.array([[r.x, r.y, r.z]
                     for r in results.face_landmarks.landmark]).flatten()            if results.face_landmarks else np.zeros(468 * 3)
    lh   = np.array([[r.x, r.y, r.z]
                     for r in results.left_hand_landmarks.landmark]).flatten()            if results.left_hand_landmarks else np.zeros(21 * 3)
    rh   = np.array([[r.x, r.y, r.z]
                     for r in results.right_hand_landmarks.landmark]).flatten()            if results.right_hand_landmarks else np.zeros(21 * 3)
    return np.concatenate([pose, face, lh, rh])   # (1662,)

def process_frame_folder(folder_path, holistic):
    """Read all frames from a folder, sorted by filename → keypoints array."""
    folder = Path(folder_path)
    imgs   = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
    imgs   = sorted(imgs, key=lambda p: p.name)   # sort by filename

    kps_list = []
    for img_path in imgs:
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        kps_list.append(extract_keypoints(results))
    return kps_list

# ── Build label map ──
unique_labels = sorted(set(s["label"] for s in raw_samples))
label_map     = {lbl: idx for idx, lbl in enumerate(unique_labels)}
id2label      = {v: k for k, v in label_map.items()}
print(f"Classes: {len(label_map)}")
print(f"Labels : {list(label_map.keys())[:10]} ...")

# ── Extract keypoints ──
processed_samples = []
skipped = 0

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    static_image_mode=True,         # ← True because we're feeding individual images
) as holistic:
    for s in tqdm(raw_samples, desc="Extracting keypoints"):
        kps_list = process_frame_folder(s["path"], holistic)
        if len(kps_list) == 0:
            skipped += 1
            continue

        seq       = np.array(kps_list, dtype=np.float32)   # (T, 1662)
        safe_name = s["label"].replace(" ", "_").replace("/", "-")
        uid       = abs(hash(s["path"])) % 100000
        save_path = os.path.join(PROCESSED_DIR, f"{safe_name}__{uid}.npy")
        np.save(save_path, seq)
        processed_samples.append({"path": save_path, "label": s["label"]})

print(f"\n✅ Done!  {len(processed_samples)} sequences saved  |  {skipped} skipped")
print(f"   Stored in: {PROCESSED_DIR}/")

json.dump(label_map,           open(CFG["label_map"], "w"), indent=2)
json.dump(processed_samples,   open(CFG["samples"],   "w"), indent=2)
print(f"   label_map → {CFG['label_map']}")
print(f"   samples   → {CFG['samples']}")


### Sanity check — visualise one keypoint sequence

In [ ]:
import matplotlib.pyplot as plt

s   = processed_samples[0]
seq = np.load(s["path"])

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

axes[0].imshow(seq.T, aspect="auto", cmap="viridis")
axes[0].set_xlabel("Frame"); axes[0].set_ylabel("Keypoint dim")
axes[0].set_title(f"Heatmap — '{s['label']}'  ({seq.shape[0]} frames)")

# Plot hand keypoint magnitude over time
lh_start, lh_end = 33*4 + 468*3, 33*4 + 468*3 + 21*3
rh_start, rh_end = lh_end, lh_end + 21*3
lh_mag = np.linalg.norm(seq[:, lh_start:lh_end].reshape(seq.shape[0], 21, 3), axis=2).mean(1)
rh_mag = np.linalg.norm(seq[:, rh_start:rh_end].reshape(seq.shape[0], 21, 3), axis=2).mean(1)
axes[1].plot(lh_mag, label="Left hand",  color="#6366F1")
axes[1].plot(rh_mag, label="Right hand", color="#F59E0B")
axes[1].set_xlabel("Frame"); axes[1].set_ylabel("Avg keypoint magnitude")
axes[1].set_title("Hand activity over time")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Sequence shape : {seq.shape}  (frames × keypoints)")
print(f"Value range    : {seq.min():.3f} → {seq.max():.3f}")


## Section 5 — PyTorch Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

MAX_LEN   = CFG["max_seq_len"]
NUM_FEATS = CFG["num_keypoints"]

class ISLDataset(Dataset):
    def __init__(self, samples, label_map):
        self.samples   = samples
        self.label_map = label_map

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        seq = np.load(s["path"])     # (T, 1662)
        T   = seq.shape[0]

        if T >= MAX_LEN:
            seq  = seq[:MAX_LEN]
            mask = np.ones(MAX_LEN,  dtype=np.float32)
        else:
            pad  = np.zeros((MAX_LEN - T, NUM_FEATS), dtype=np.float32)
            seq  = np.concatenate([seq, pad], axis=0)
            mask = np.array([1.]*T + [0.]*(MAX_LEN-T), dtype=np.float32)

        label = self.label_map[s["label"]]
        return (
            torch.tensor(seq,   dtype=torch.float32),
            torch.tensor(mask,  dtype=torch.float32),
            torch.tensor(label, dtype=torch.long),
        )

# ── Reload saved metadata (safe to run independently after Section 4) ──
processed_samples = json.load(open(CFG["samples"]))
label_map         = json.load(open(CFG["label_map"]))
id2label          = {v: k for k, v in label_map.items()}
NUM_CLASSES       = len(label_map)

labels_all = [s["label"] for s in processed_samples]

train_s, temp_s = train_test_split(processed_samples, test_size=0.30,
                                   stratify=labels_all, random_state=42)
val_s, test_s   = train_test_split(temp_s, test_size=0.50,
                                   stratify=[s["label"] for s in temp_s],
                                   random_state=42)

BS = CFG["batch_size"]
train_loader = DataLoader(ISLDataset(train_s, label_map), batch_size=BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ISLDataset(val_s,   label_map), batch_size=BS, shuffle=False, num_workers=0)
test_loader  = DataLoader(ISLDataset(test_s,  label_map), batch_size=BS, shuffle=False, num_workers=0)

print(f"✅ Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")
print(f"   Classes: {NUM_CLASSES}")
seqs, masks, labels = next(iter(train_loader))
print(f"   Batch — seqs {tuple(seqs.shape)} | masks {tuple(masks.shape)} | labels {tuple(labels.shape)}")


## Section 6 — Model

In [ ]:
import torch.nn as nn

class ISL_BiLSTM(nn.Module):
    """Bidirectional LSTM with temporal attention for ISL classification."""
    def __init__(self, input_size=1662, num_classes=100):
        super().__init__()
        h  = CFG["hidden_size"]
        L  = CFG["num_layers"]
        dp = CFG["dropout"]

        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=h, num_layers=L,
                            batch_first=True, bidirectional=True,
                            dropout=dp if L > 1 else 0.0)
        self.attn    = nn.Linear(h * 2, 1)
        self.dropout = nn.Dropout(dp)
        self.fc      = nn.Linear(h * 2, num_classes)

    def forward(self, x, mask=None):
        x       = self.input_norm(x)
        out, _  = self.lstm(x)                              # (B, T, 2H)
        scores  = self.attn(out).squeeze(-1)                # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = (out * weights).sum(dim=1)                # (B, 2H)
        return self.fc(self.dropout(context))

DEVICE = torch.device(CFG["device"] if torch.cuda.is_available() else "cpu")
model  = ISL_BiLSTM(input_size=NUM_FEATS, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model  : ISL_BiLSTM")
print(f"   Device : {DEVICE}")
print(f"   Params : {total_params:,}")


## Section 7 — Train

In [ ]:
from tqdm.notebook import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

history = {"train_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, patience_ctr = 0.0, 0

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for seqs, masks, lbls in tqdm(train_loader, desc=f"Ep {epoch:03d} [train]", leave=False):
        seqs, masks, lbls = seqs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(seqs, masks)
        loss = criterion(out, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == lbls).sum().item()
        total      += lbls.size(0)

    train_acc = correct / total
    avg_loss  = total_loss / len(train_loader)

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for seqs, masks, lbls in val_loader:
            seqs, masks = seqs.to(DEVICE), masks.to(DEVICE)
            out = model(seqs, masks)
            vc += (out.argmax(1) == lbls.to(DEVICE)).sum().item()
            vt += lbls.size(0)
    val_acc = vc / vt
    scheduler.step()

    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    indicator = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save({"model_state": model.state_dict(), "label_map": label_map,
                    "config": CFG}, CFG["checkpoint"])
        indicator = "  ← best ✓"
    else:
        patience_ctr += 1
        if patience_ctr >= CFG["patience"]:
            print("\n⏹ Early stopping.")
            break

    print(f"Ep {epoch:03d} | loss {avg_loss:.4f} | train {train_acc:.3f} | val {val_acc:.3f}{indicator}")

print(f"\n✅ Best val accuracy: {best_val_acc:.4f}")


### Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["train_loss"], color="#3B82F6", label="Train loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history["train_acc"], color="#10B981", label="Train acc")
axes[1].plot(history["val_acc"],   color="#F59E0B", label="Val acc")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()


## Section 8 — Evaluate on test set

In [ ]:
from jiwer import wer as compute_wer
import seaborn as sns
from sklearn.metrics import confusion_matrix

ckpt = torch.load(CFG["checkpoint"], map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

refs, hyps = [], []
with torch.no_grad():
    for seqs, masks, lbls in tqdm(test_loader, desc="Testing"):
        out   = model(seqs.to(DEVICE), masks.to(DEVICE))
        preds = out.argmax(1).cpu().tolist()
        for p, t in zip(preds, lbls.tolist()):
            hyps.append(id2label[p])
            refs.append(id2label[t])

acc     = sum(h==r for h,r in zip(hyps,refs)) / len(refs)
word_er = compute_wer(refs, hyps)

print(f"{'='*45}")
print(f"  Test accuracy  : {acc:.4f}  ({int(acc*len(refs))}/{len(refs)})")
print(f"  Word error rate: {word_er:.4f}")
print(f"{'='*45}")


In [ ]:
# Confusion matrix
cm = confusion_matrix([label_map[r] for r in refs],
                      [label_map[h] for h in hyps])
names = [id2label[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(max(8, NUM_CLASSES*0.4), max(6, NUM_CLASSES*0.4)))
sns.heatmap(cm, xticklabels=names, yticklabels=names,
            cmap="Blues", fmt="d", annot=(NUM_CLASSES<=20))
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion matrix — test set")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()


## Section 9 — Inference
**9A** — predict from a folder of frames  
**9B** — predict from a video file  
**9C** — live webcam  
**9D** — Gradio web UI


### 9A — Predict from a frame folder

In [ ]:
def predict_from_frames(folder_path):
    """Given a folder of ordered frames, return top-5 predictions."""
    kps_list = []
    with mp_holistic.Holistic(min_detection_confidence=0.5, static_image_mode=True) as h:
        for img_path in sorted(Path(folder_path).glob("*.jpg")):
            frame   = cv2.imread(str(img_path))
            if frame is None: continue
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = h.process(rgb)
            kps_list.append(extract_keypoints(results))

    if not kps_list:
        return None, "No frames found."

    seq = np.array(kps_list[:MAX_LEN], dtype=np.float32)
    T   = seq.shape[0]
    if T < MAX_LEN:
        seq = np.concatenate([seq, np.zeros((MAX_LEN-T, NUM_FEATS))], axis=0)

    model.eval()
    seq_t  = torch.tensor(seq).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, MAX_LEN).to(DEVICE)
    with torch.no_grad():
        out   = model(seq_t, mask_t)
        probs = out.softmax(1)[0]
        top5  = probs.topk(min(5, NUM_CLASSES))

    top_label = id2label[out.argmax(1).item()]
    ranked    = [(id2label[i.item()], f"{s.item()*100:.1f}%")
                 for s, i in zip(top5.values, top5.indices)]
    return top_label, ranked

# ── Usage ──
TEST_FOLDER = "path/to/a/frame/folder"   # ← change this to any folder from your dataset
if os.path.exists(TEST_FOLDER):
    top, ranked = predict_from_frames(TEST_FOLDER)
    print(f"🏆 Predicted: {top}")
    for lbl, conf in ranked:
        print(f"  {lbl:<35} {conf}")
else:
    print(f"Folder not found. Update TEST_FOLDER to point to any frame folder.")


### 9B — Predict from a video file

In [ ]:
def predict_from_video(video_path):
    kps_list = []
    cap = cv2.VideoCapture(str(video_path))
    with mp_holistic.Holistic(min_detection_confidence=0.5) as h:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = h.process(rgb)
            kps_list.append(extract_keypoints(results))
    cap.release()
    if not kps_list:
        return None, "No frames in video."
    seq = np.array(kps_list[:MAX_LEN], dtype=np.float32)
    T   = seq.shape[0]
    if T < MAX_LEN:
        seq = np.concatenate([seq, np.zeros((MAX_LEN-T, NUM_FEATS))], axis=0)
    model.eval()
    seq_t  = torch.tensor(seq).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, MAX_LEN).to(DEVICE)
    with torch.no_grad():
        out  = model(seq_t, mask_t)
        probs = out.softmax(1)[0]
        top5  = probs.topk(min(5, NUM_CLASSES))
    top_label = id2label[out.argmax(1).item()]
    ranked    = [(id2label[i.item()], f"{s.item()*100:.1f}%")
                 for s, i in zip(top5.values, top5.indices)]
    return top_label, ranked

VIDEO_PATH = "path/to/video.mp4"   # ← update this
if os.path.exists(VIDEO_PATH):
    top, ranked = predict_from_video(VIDEO_PATH)
    print(f"🏆 {top}")
    for l,c in ranked: print(f"  {l:<35} {c}")
else:
    print("Update VIDEO_PATH above.")


### 9C — Live webcam
Press **Q** to stop.

In [ ]:
def run_webcam():
    model.eval()
    cap      = cv2.VideoCapture(0)
    sequence = []
    pred     = "Waiting..."
    conf     = 0.0

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as h:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = h.process(rgb)
            sequence.append(extract_keypoints(results))

            if len(sequence) == MAX_LEN:
                seq_arr = np.array(sequence, dtype=np.float32)
                seq_t   = torch.tensor(seq_arr).unsqueeze(0).to(DEVICE)
                mask_t  = torch.ones(1, MAX_LEN).to(DEVICE)
                with torch.no_grad():
                    out   = model(seq_t, mask_t)
                    probs = out.softmax(1)[0]
                pred = id2label[out.argmax(1).item()]
                conf = probs[out.argmax(1).item()].item()
                sequence = sequence[10:]   # slide window by 10 frames

            mp_draw = mp.solutions.drawing_utils
            mp_draw.draw_landmarks(frame, results.left_hand_landmarks,
                                   mp.solutions.holistic.HAND_CONNECTIONS)
            mp_draw.draw_landmarks(frame, results.right_hand_landmarks,
                                   mp.solutions.holistic.HAND_CONNECTIONS)

            bar = int(conf * 300)
            cv2.rectangle(frame, (10,60), (10+bar,80), (0,200,100), -1)
            cv2.rectangle(frame, (10,60), (310,80), (200,200,200), 1)
            cv2.putText(frame, f"Prediction: {pred}",       (10,45),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,230,100), 2)
            cv2.putText(frame, f"Confidence: {conf*100:.1f}%", (10,100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230,230,230), 1)
            cv2.putText(frame, f"Buffer: {len(sequence)}/{MAX_LEN}", (10,120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)
            cv2.imshow("ISL Translation  |  Q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    cap.release()
    cv2.destroyAllWindows()

run_webcam()


### 9D — Gradio web UI

In [ ]:
import gradio as gr

def gradio_video(video_path):
    if not video_path: return "No video uploaded."
    top, ranked = predict_from_video(video_path)
    out  = f"### 🏆 Predicted: `{top}`\n\n**Top predictions:**\n"
    out += "\n".join(f"- `{l}` — {c}" for l,c in ranked)
    return out

def gradio_frames(frame_folder):
    if not frame_folder or not os.path.isdir(frame_folder):
        return "Enter a valid folder path containing .jpg frames."
    top, ranked = predict_from_frames(frame_folder)
    out  = f"### 🏆 Predicted: `{top}`\n\n**Top predictions:**\n"
    out += "\n".join(f"- `{l}` — {c}" for l,c in ranked)
    return out

with gr.Blocks(title="ISL Translator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤟 Indian Sign Language Translator\nUpload a video or enter a frames folder path.")
    with gr.Tab("Upload video"):
        vid_in  = gr.Video(label="ISL video")
        vid_out = gr.Markdown()
        gr.Button("Predict").click(gradio_video, vid_in, vid_out)
    with gr.Tab("Frame folder"):
        fold_in  = gr.Textbox(label="Path to frame folder", placeholder="C:/ISL_CSLRT_Corpus/Frames_Sentence_Level/...")
        fold_out = gr.Markdown()
        gr.Button("Predict").click(gradio_frames, fold_in, fold_out)

demo.launch(inbrowser=True, share=False)


---
## Quick troubleshooting

| Problem | Fix |
|---|---|
| `Column not found` in Section 3 | Print `df.columns` and update `COL_LABEL` |
| `0 matches` in `find_frame_folders` | Set `USE_FALLBACK = True` in the next cell |
| `mediapipe` install error | `pip install mediapipe==0.10.11` in terminal |
| CUDA out of memory | Reduce `batch_size` to 8 or set `device: cpu` |
| OpenCV window doesn't open | Run `run_webcam()` in a `.py` file, not Jupyter |
